In [1]:
import os
from dataclasses import dataclass
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_model, set_trainable
import tensorflow as tf
import keras

I0000 00:00:1793747475.842633   37537 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793747475.907075   37537 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793747477.521358   37537 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/nima/Projects/ML-Project-Casting_Classification/Project'

In [5]:
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path

@dataclass(frozen=True)
class PrepareBaseModelParams:    
    IMG_SIZE: list
    BATCH_SIZE : int
    INCLUDE_TOP: bool
    TRAINING : bool
    LEARNINGRATE_P1: float
    LEARNINGRATE_P2: float
    DROPOUT : float    
    ROTATION_FACTOR : float
    BRIGHTNESS_FACTOR : float
    CONTRAST_FACTOR : float

In [6]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.prepare_base_model.root_dir])


    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir = Path(config.root_dir),
            base_model_path = Path(config.base_model_path),
            updated_base_model_path = Path(config.updated_base_model_path)            
        )
        return prepare_base_model_config

    
    def get_prepare_base_model_params(self) -> PrepareBaseModelParams:
        params = self.params
        
        prepare_base_model_params = PrepareBaseModelParams(
            IMG_SIZE=params.IMG_SIZE,
            BATCH_SIZE=params.BATCH_SIZE,
            INCLUDE_TOP=params.INCLUDE_TOP,
            TRAINING=params.TRAINING,
            LEARNINGRATE_P1=params.LEARNINGRATE_P1,
            LEARNINGRATE_P2=params.LEARNINGRATE_P2,
            DROPOUT=params.DROPOUT,
            ROTATION_FACTOR=params.ROTATION_FACTOR,
            BRIGHTNESS_FACTOR=params.BRIGHTNESS_FACTOR,
            CONTRAST_FACTOR=params.CONTRAST_FACTOR
            )
        return prepare_base_model_params

In [7]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig, params: PrepareBaseModelParams):
        self.config = config
        self.params = params

    def get_base_model(self):
        self.model = tf.keras.applications.MobileNetV2(
            input_shape=(*self.params.IMG_SIZE, 3),   # the base always sees 3 channels
            weights="imagenet",
            include_top=self.params.INCLUDE_TOP,      # must be False
        )
        logger.info("base model is created")
        save_model(path=self.config.base_model_path, model=self.model)

    def update_base_model(self):
        """Wrap the frozen base in grayscale input + head. Saved UNCOMPILED: compiling is phase-specific (04)."""
        set_trainable(self.model, freeze_all=True)

        inputs = keras.Input(shape=(*self.params.IMG_SIZE, 1), name="gray_input")
        x = keras.layers.Concatenate(name="gray_to_rgb")([inputs, inputs, inputs])
        x = keras.layers.Rescaling(scale=2.0, offset=-1.0, name="to_minus1_1")(x)
        x = self.model(x, training=False)
        x = keras.layers.GlobalAveragePooling2D()(x)
        x = keras.layers.Dropout(self.params.DROPOUT)(x)
        outputs = keras.layers.Dense(1, activation="sigmoid", name="defect_prob")(x)

        self.full_model = keras.Model(inputs, outputs, name="updated_mobilenetv2")
        self.full_model.summary()
        logger.info("updated base model is created")
        save_model(path=self.config.updated_base_model_path, model=self.full_model)

In [8]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model_params = config.get_prepare_base_model_params()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config, params=prepare_base_model_params)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-11-04 00:11:18,781: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-11-04 00:11:18,785: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-04 00:11:18,786: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-11-04 00:11:18,787: INFO: common: created directory at: artifacts/prepare_base_model]


E0000 00:00:1793747478.824174   37537 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1793747478.824687   37573 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1793747478.838030   37537 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


[2026-11-04 00:11:19,990: INFO: 1916286337: base model is created]


Model: "updated_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gray_input          │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gray_to_rgb         │ (None, 224, 224,  │          0 │ gray_input[0][0], │
│ (Concatenate)       │ 3)                │            │ gray_input[0][0], │
│                     │                   │            │ gray_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ to_minus1_1         │ (None, 224, 224,  │          0 │ gray_to_rgb[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ to_minus1_1[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1280)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ defect_prob (Dense) │ (None, 1)         │      1,281 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

[2026-11-04 00:11:20,503: INFO: 1916286337: updated base model is created]
